# Notebook 4: Feature Engineering

## Introduction

The exploratory data analysis demonstrated that Delhi's air quality is influenced by the combined effects of regional biomass burning and meteorological conditions. However, the original variables alone do not fully capture the temporal dependencies required for accurate next-day AQI prediction.

This notebook transforms the cleaned dataset into a modelling-ready dataset by engineering meaningful features that represent historical pollution trends, fire activity, meteorological conditions, and temporal patterns. These engineered features are designed to improve predictive performance while preserving the interpretability of the final machine learning models.

The feature engineering process consists of the following stages:

- Temporal Feature Engineering
- Historical AQI Feature Engineering
- Fire Feature Engineering
- Wind Feature Engineering
- Target Variable Creation
- Missing Value Handling
- Final Feature Validation

# 1. Load Dataset and Initial Validation

The cleaned master dataset is loaded and prepared for feature engineering. Before creating new features, the data types and structure are verified to ensure that the dataset is ready for further processing.

In [15]:
import pandas as pd
import numpy as np

master_df = pd.read_csv(
    r"Master_cleaned_dataset/master_dataset.csv"
)

master_df["Date"] = pd.to_datetime(master_df["Date"])

feature_df = master_df.copy()

In [16]:
feature_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1851 entries, 0 to 1850
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Date               1851 non-null   datetime64[ns]
 1   AQI                1851 non-null   float64       
 2   AQI_Bucket         1851 non-null   object        
 3   PM2.5              1851 non-null   float64       
 4   Fire_Count         1851 non-null   int64         
 5   Total_FRP          1851 non-null   float64       
 6   Mean_FRP           1851 non-null   float64       
 7   Max_FRP            1851 non-null   float64       
 8   Mean_U10           1851 non-null   float64       
 9   Mean_V10           1851 non-null   float64       
 10  Mean_Wind_Speed    1851 non-null   float64       
 11  Max_Wind_Speed     1851 non-null   float64       
 12  Avg_Temperature_C  1851 non-null   float64       
 13  Avg_DewPoint_C     1851 non-null   float64       
 14  Relative

# 2. Temporal Feature Engineering

The exploratory data analysis identified strong seasonal and temporal patterns in Delhi's air quality. Therefore, temporal features are extracted from the Date column to enable the machine learning models to capture recurring monthly, weekly, and seasonal variations.

In [17]:
feature_df["Month"] = feature_df["Date"].dt.month

feature_df["Day_of_Week"] = feature_df["Date"].dt.dayofweek

In [18]:
def get_season(month):

    if month in [12, 1, 2]:
        return "Winter"

    elif month in [3, 4, 5]:
        return "Summer"

    elif month in [6, 7, 8, 9]:
        return "Monsoon"

    else:
        return "Post-Monsoon"


feature_df["Season"] = feature_df["Month"].apply(get_season)

In [19]:
season_mapping = {
    "Winter": 0,
    "Summer": 1,
    "Monsoon": 2,
    "Post-Monsoon": 3
}

feature_df["Season"] = feature_df["Season"].map(season_mapping)

### Validation

The engineered temporal features are verified to ensure that they have been generated correctly before proceeding with historical feature engineering.

In [20]:
feature_df[
    [
        "Date",
        "Month",
        "Day_of_Week",
        "Season"
    ]
].head()

,Date,Month,Day_of_Week,Season
0,2015-01-01,1,3,0
1,2015-01-02,1,4,0
2,2015-01-03,1,5,0
3,2015-01-04,1,6,0
4,2015-01-05,1,0,0


In [21]:
feature_df[
    [
        "Month",
        "Day_of_Week",
        "Season"
    ]
].describe()

,Month,Day_of_Week,Season
count,1851.000000,1851.000000,1851.000000
mean,6.177742,3.003782,1.332253
std,3.527385,2.001752,1.043555
min,1.000000,0.000000,0.000000
25%,3.000000,1.000000,0.000000
50%,6.000000,3.000000,1.000000
75%,9.000000,5.000000,2.000000
max,12.000000,6.000000,3.000000


# 3. Historical AQI Feature Engineering

Air quality exhibits strong temporal persistence, meaning that pollution levels on a given day are influenced by those of preceding days. The exploratory data analysis revealed gradual day-to-day changes in AQI, indicating that recent historical observations provide valuable information for predicting next-day air quality.

To capture these short-term temporal dependencies, lag-based and rolling statistical features are created using only past AQI values. A three-day historical window is selected to represent recent pollution behaviour while avoiding unnecessary model complexity and preventing data leakage.

## 3.1 AQI Lag Features

Lag features represent AQI observations from previous days. These features enable the model to learn how recent pollution levels influence future air quality.

In [22]:
feature_df["AQI_lag1"] = feature_df["AQI"].shift(1)

feature_df["AQI_lag2"] = feature_df["AQI"].shift(2)

feature_df["AQI_lag3"] = feature_df["AQI"].shift(3)

## 3.2 Rolling AQI Features

Rolling statistics summarize recent pollution behaviour by considering multiple previous observations. The rolling mean captures the average AQI over the preceding three days, while the rolling standard deviation measures short-term variability in pollution levels.

The rolling window is shifted by one day to ensure that only historical information is used for prediction.

In [23]:
feature_df["AQI_rolling_mean_3"] = (
    feature_df["AQI"]
    .shift(1)
    .rolling(window=3)
    .mean()
)

feature_df["AQI_rolling_std_3"] = (
    feature_df["AQI"]
    .shift(1)
    .rolling(window=3)
    .std()
)

### Validation

The newly engineered AQI features are examined to verify that lag values and rolling statistics have been generated correctly before proceeding with additional feature engineering.

In [24]:
feature_df[
    [
        "Date",
        "AQI",
        "AQI_lag1",
        "AQI_lag2",
        "AQI_lag3",
        "AQI_rolling_mean_3",
        "AQI_rolling_std_3"
    ]
].head(10)

,Date,AQI,AQI_lag1,AQI_lag2,AQI_lag3,AQI_rolling_mean_3,AQI_rolling_std_3
0,2015-01-01,472.0,NaN,NaN,NaN,NaN,NaN
1,2015-01-02,454.0,472.0,NaN,NaN,NaN,NaN
2,2015-01-03,143.0,454.0,472.0,NaN,NaN,NaN
3,2015-01-04,319.0,143.0,454.0,472.0,356.333333,184.971169
4,2015-01-05,325.0,319.0,143.0,454.0,305.333333,155.949778
5,2015-01-06,318.0,325.0,319.0,143.0,262.333333,103.389232
6,2015-01-07,353.0,318.0,325.0,319.0,320.666667,3.785939
7,2015-01-08,383.0,353.0,318.0,325.0,332.000000,18.520259
8,2015-01-09,375.0,383.0,353.0,318.0,351.333333,32.532035
9,2015-01-10,376.0,375.0,383.0,353.0,370.333333,15.534907


In [25]:
feature_df[
    [
        "AQI_lag1",
        "AQI_lag2",
        "AQI_lag3",
        "AQI_rolling_mean_3",
        "AQI_rolling_std_3"
    ]
].isnull().sum()

AQI_lag1              1
AQI_lag2              2
AQI_lag3              3
AQI_rolling_mean_3    3
AQI_rolling_std_3     3
dtype: int64

# 4. Fire Feature Engineering

The exploratory data analysis demonstrated that regional biomass burning is positively associated with Delhi's air quality, although the relationship is weaker than that of historical AQI. Since the impact of biomass burning may persist beyond a single day, lag-based fire features are created to capture short-term delayed effects of fire activity on air quality.

Both Fire Count and Total Fire Radiative Power (FRP) are considered to represent the frequency and intensity of biomass burning, respectively.

## 4.1 Fire Count Lag Features

Fire Count lag features represent the number of fire events observed during the previous days. These variables enable the model to account for delayed impacts of regional biomass burning on subsequent air quality.

In [27]:
feature_df["Fire_Count_lag1"] = feature_df["Fire_Count"].shift(1)

feature_df["Fire_Count_lag2"] = feature_df["Fire_Count"].shift(2)

## 4.2 Total Fire Radiative Power (FRP) Lag Features

Lag features are also created for Total Fire Radiative Power (FRP), which represents the cumulative intensity of biomass burning. These features allow the model to incorporate recent fire intensity while avoiding the use of future information.

In [28]:
feature_df["Total_FRP_lag1"] = feature_df["Total_FRP"].shift(1)

feature_df["Total_FRP_lag2"] = feature_df["Total_FRP"].shift(2)

### Validation

The engineered fire-related lag features are verified to ensure that previous-day fire activity and fire intensity have been correctly incorporated into the dataset before proceeding with additional feature engineering.

In [29]:
feature_df[
    [
        "Date",
        "Fire_Count",
        "Fire_Count_lag1",
        "Fire_Count_lag2",
        "Total_FRP",
        "Total_FRP_lag1",
        "Total_FRP_lag2"
    ]
].head(10)

,Date,Fire_Count,Fire_Count_lag1,Fire_Count_lag2,Total_FRP,Total_FRP_lag1,Total_FRP_lag2
0,2015-01-01,30,NaN,NaN,129.63,NaN,NaN
1,2015-01-02,15,30.0,NaN,84.92,129.63,NaN
2,2015-01-03,4,15.0,30.0,5.93,84.92,129.63
3,2015-01-04,15,4.0,15.0,81.51,5.93,84.92
4,2015-01-05,34,15.0,4.0,110.50,81.51,5.93
5,2015-01-06,27,34.0,15.0,78.98,110.50,81.51
6,2015-01-07,5,27.0,34.0,14.42,78.98,110.50
7,2015-01-08,10,5.0,27.0,62.42,14.42,78.98
8,2015-01-09,20,10.0,5.0,124.13,62.42,14.42
9,2015-01-10,125,20.0,10.0,460.38,124.13,62.42


In [30]:
feature_df[
    [
        "Fire_Count_lag1",
        "Fire_Count_lag2",
        "Total_FRP_lag1",
        "Total_FRP_lag2"
    ]
].isnull().sum()

Fire_Count_lag1    1
Fire_Count_lag2    2
Total_FRP_lag1     1
Total_FRP_lag2     2
dtype: int64

# 5. Target Variable Creation

The objective of this study is to predict the Air Quality Index (AQI) for the following day. Therefore, the target variable is created by shifting the AQI values one day forward.

This approach ensures that the model learns to predict future air quality using only information available up to the current day, thereby preventing target leakage.

## 5.1 Next-Day AQI

The target variable, **AQI_next_day**, represents the AQI value observed on the following day. During model training, all engineered features from the current day will be used to predict this target.

In [31]:
feature_df["AQI_next_day"] = feature_df["AQI"].shift(-1)

### Validation

The target variable is verified to ensure that each observation correctly corresponds to the AQI recorded on the following day.

In [32]:
feature_df[
    [
        "Date",
        "AQI",
        "AQI_next_day"
    ]
].head(10)

,Date,AQI,AQI_next_day
0,2015-01-01,472.0,454.0
1,2015-01-02,454.0,143.0
2,2015-01-03,143.0,319.0
3,2015-01-04,319.0,325.0
4,2015-01-05,325.0,318.0
5,2015-01-06,318.0,353.0
6,2015-01-07,353.0,383.0
7,2015-01-08,383.0,375.0
8,2015-01-09,375.0,376.0
9,2015-01-10,376.0,379.0


In [33]:
feature_df["AQI_next_day"].isnull().sum()

np.int64(1)

# 6. Missing Value Handling

The creation of lag features, rolling statistics, and the target variable naturally introduces missing values at the beginning and end of the dataset. Since these observations do not contain sufficient historical or future information for prediction, they are removed before model development.

In [34]:
feature_df.isnull().sum()

Date                  0
AQI                   0
AQI_Bucket            0
PM2.5                 0
Fire_Count            0
Total_FRP             0
Mean_FRP              0
Max_FRP               0
Mean_U10              0
Mean_V10              0
Mean_Wind_Speed       0
Max_Wind_Speed        0
Avg_Temperature_C     0
Avg_DewPoint_C        0
Relative_Humidity     0
Month                 0
Day_of_Week           0
Season                0
AQI_lag1              1
AQI_lag2              2
AQI_lag3              3
AQI_rolling_mean_3    3
AQI_rolling_std_3     3
Fire_Count_lag1       1
Fire_Count_lag2       2
Total_FRP_lag1        1
Total_FRP_lag2        2
AQI_next_day          1
dtype: int64

In [35]:
feature_df = feature_df.dropna().reset_index(drop=True)

In [36]:
feature_df.isnull().sum()

Date                  0
AQI                   0
AQI_Bucket            0
PM2.5                 0
Fire_Count            0
Total_FRP             0
Mean_FRP              0
Max_FRP               0
Mean_U10              0
Mean_V10              0
Mean_Wind_Speed       0
Max_Wind_Speed        0
Avg_Temperature_C     0
Avg_DewPoint_C        0
Relative_Humidity     0
Month                 0
Day_of_Week           0
Season                0
AQI_lag1              0
AQI_lag2              0
AQI_lag3              0
AQI_rolling_mean_3    0
AQI_rolling_std_3     0
Fire_Count_lag1       0
Fire_Count_lag2       0
Total_FRP_lag1        0
Total_FRP_lag2        0
AQI_next_day          0
dtype: int64

# 7. Final Feature Validation

Following feature engineering and missing value handling, the final modelling dataset is examined to verify its completeness, structure, and suitability for subsequent statistical analysis and machine learning.

The validation includes checking the dataset dimensions, data types, missing values, and the final set of engineered features.

## 7.1 Dataset Dimensions

The number of observations and variables is examined to confirm that the dataset has been prepared successfully after feature engineering.

In [37]:
print("Dataset Shape:", feature_df.shape)

Dataset Shape: (1847, 28)


## 7.2 Data Types

The data types of all variables are verified to ensure compatibility with the statistical analyses and machine learning models developed in the subsequent notebooks.

In [38]:
feature_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1847 entries, 0 to 1846
Data columns (total 28 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Date                1847 non-null   datetime64[ns]
 1   AQI                 1847 non-null   float64       
 2   AQI_Bucket          1847 non-null   object        
 3   PM2.5               1847 non-null   float64       
 4   Fire_Count          1847 non-null   int64         
 5   Total_FRP           1847 non-null   float64       
 6   Mean_FRP            1847 non-null   float64       
 7   Max_FRP             1847 non-null   float64       
 8   Mean_U10            1847 non-null   float64       
 9   Mean_V10            1847 non-null   float64       
 10  Mean_Wind_Speed     1847 non-null   float64       
 11  Max_Wind_Speed      1847 non-null   float64       
 12  Avg_Temperature_C   1847 non-null   float64       
 13  Avg_DewPoint_C      1847 non-null   float64     

## 7.3 Missing Value Verification

The dataset is examined once more to confirm that no missing values remain after removing observations affected by lag feature generation.

In [39]:
feature_df.isnull().sum()

Date                  0
AQI                   0
AQI_Bucket            0
PM2.5                 0
Fire_Count            0
Total_FRP             0
Mean_FRP              0
Max_FRP               0
Mean_U10              0
Mean_V10              0
Mean_Wind_Speed       0
Max_Wind_Speed        0
Avg_Temperature_C     0
Avg_DewPoint_C        0
Relative_Humidity     0
Month                 0
Day_of_Week           0
Season                0
AQI_lag1              0
AQI_lag2              0
AQI_lag3              0
AQI_rolling_mean_3    0
AQI_rolling_std_3     0
Fire_Count_lag1       0
Fire_Count_lag2       0
Total_FRP_lag1        0
Total_FRP_lag2        0
AQI_next_day          0
dtype: int64

## 7.4 Final Feature List

The complete list of original and engineered features is displayed to verify the final modelling dataset before export.

In [40]:
feature_df.columns.tolist()

['Date',
 'AQI',
 'AQI_Bucket',
 'PM2.5',
 'Fire_Count',
 'Total_FRP',
 'Mean_FRP',
 'Max_FRP',
 'Mean_U10',
 'Mean_V10',
 'Mean_Wind_Speed',
 'Max_Wind_Speed',
 'Avg_Temperature_C',
 'Avg_DewPoint_C',
 'Relative_Humidity',
 'Month',
 'Day_of_Week',
 'Season',
 'AQI_lag1',
 'AQI_lag2',
 'AQI_lag3',
 'AQI_rolling_mean_3',
 'AQI_rolling_std_3',
 'Fire_Count_lag1',
 'Fire_Count_lag2',
 'Total_FRP_lag1',
 'Total_FRP_lag2',
 'AQI_next_day']

# 8. Save Final Modelling Dataset

The engineered dataset is exported for use in the subsequent statistical analysis and machine learning notebooks.

In [42]:
feature_df.to_csv(
    "feature_engineered_dataset.csv",
    index=False
)

print("Feature engineered dataset saved successfully.")

Feature engineered dataset saved successfully.


# 9. Feature Engineering Summary

This notebook transformed the cleaned master dataset into a modelling-ready dataset through the creation of meaningful temporal, historical, and fire-related features.

The major feature engineering steps included:

- Extraction of temporal features (Month, Day of Week, and Season)
- Creation of historical AQI lag and rolling statistical features
- Engineering of lag-based fire activity and fire intensity features
- Construction of the next-day AQI target variable
- Removal of observations containing incomplete historical information
- Validation of the final modelling dataset

The resulting dataset preserves only information available up to the prediction date, thereby preventing data leakage while capturing the temporal and environmental characteristics necessary for next-day AQI prediction.

This feature-engineered dataset forms the foundation for the subsequent statistical analysis and machine learning modelling stages.